#### 02.Station-Static Features
In this section, we will create station-specific features, save them to a small table, and later join it with other features for training.
We construct the following features,
* `t0_x_km`, `t0_y_km`: The relative location of a station in kilometers
* `t0_wet_ratio`: compared to the islandwise average, relative strength of rainfall (y_mm > 0mm) of a station in the next 30 minutes.
* `t0_heavy_ratio`: definition similar to the above, yet the threshold of rainfall replaced by y_mm > 5mm.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
MATRIX = ROOT / "task1_revision" / "data" / "matrix_5min"
OUT = ROOT / "task2" / "data"
assert (OUT / "labels").exists(), OUT

sys.path.insert(0, "../code")
import config as C

In [2]:
from pyspark.sql import SparkSession, functions as F, Window

spark = C.get_spark("task2-station-static")

# Read our processed data
labels = spark.read.parquet(str(OUT / "labels"))
labels.show(10)

+----------+------+-------------------+----+-----+---------+-------+----+
|station_id|  slot|                 ts|y_mm|label|past30_mm| regime|year|
+----------+------+-------------------+----+-----+---------+-------+----+
|      S112|525888|2022-01-01 00:00:00| 0.0|  0.0|      0.0|dry_now|2022|
|      S112|525889|2022-01-01 00:05:00| 0.0|  0.0|      0.0|dry_now|2022|
|      S112|525890|2022-01-01 00:10:00| 0.0|  0.0|      0.0|dry_now|2022|
|      S112|525891|2022-01-01 00:15:00| 0.0|  0.0|      0.0|dry_now|2022|
|      S112|525892|2022-01-01 00:20:00| 0.0|  0.0|      0.0|dry_now|2022|
|      S112|525893|2022-01-01 00:25:00| 0.0|  0.0|      0.0|dry_now|2022|
|      S112|525894|2022-01-01 00:30:00| 0.0|  0.0|      0.0|dry_now|2022|
|      S112|525895|2022-01-01 00:35:00| 0.0|  0.0|      0.0|dry_now|2022|
|      S112|525896|2022-01-01 00:40:00| 0.0|  0.0|      0.0|dry_now|2022|
|      S112|525897|2022-01-01 00:45:00| 0.0|  0.0|      0.0|dry_now|2022|
+----------+------+-------------------

In [3]:
# we will first construct the feature O/E ratio, representing the relative strength of rainfall 
# of a certain station compared to the islandwise average performance
tr = (labels.filter(F.col("year").between(2017, 2022))
            .withColumn("wet",   (F.col("label") > 0).cast("double"))
            .withColumn("heavy", (F.col("label") >= 6).cast("double")))   

net = tr.groupBy("slot").agg(F.avg("wet").alias("p_wet"),
                             F.avg("heavy").alias("p_heavy"))

# Here, we add a shrinkage term M=500 on both numerator and denominator, because the ratio O/E has a 
# extremely large span, applying shrinkage will help normalize stations with few observations 
M = 500
stn = (tr.join(net, on="slot")
         .groupBy("station_id")
         .agg(F.count("*").alias("n"),
              F.sum("wet").alias("O_wet"),     F.sum("p_wet").alias("E_wet"),
              F.sum("heavy").alias("O_heavy"), F.sum("p_heavy").alias("E_heavy"))
         .withColumn("t0_wet_ratio",   (F.col("O_wet") + M) / (F.col("E_wet") + M))
         .withColumn("t0_heavy_ratio", (F.col("O_heavy") + M) / (F.col("E_heavy") + M)))

In [4]:
pdf = stn.toPandas()
coords = pd.read_csv(MATRIX / "stations.csv")
pdf = pdf.merge(coords[["station_id", "location_longitude", "location_latitude"]], on="station_id")

LAT0, LON0 = 1.35, 103.82        
pdf["t0_x_km"] = (pdf.location_longitude - LON0) * 111.32 * np.cos(np.radians(LAT0))
pdf["t0_y_km"] = (pdf.location_latitude - LAT0) * 110.57

pdf.to_csv(OUT / "station_static.csv", index=False)

In [5]:
pdf

,station_id,n,O_wet,E_wet,O_heavy,E_heavy,t0_wet_ratio,t0_heavy_ratio,location_longitude,location_latitude,t0_x_km,t0_y_km
0,S208,262999,12772.0,17239.276424,1503.0,2361.754526,0.748170,0.699920,104.00317,1.31360,20.384825,-4.024748
1,S117,361958,20023.0,20600.288204,2674.0,2629.126322,0.972641,1.014341,103.67900,1.25600,-15.691763,-10.393580
2,S118,569068,32061.0,33282.129761,4333.0,4503.983591,0.963853,0.965831,103.84610,1.29940,2.904646,-5.594842
3,S77,585786,35026.0,34917.357685,4657.0,4647.152895,1.003067,1.001913,103.81250,1.29370,-0.834668,-6.225091
4,S205,233535,14777.0,14910.836813,2032.0,2078.323916,0.991315,0.982033,103.91160,1.38829,10.194082,4.233725
...,...,...,...,...,...,...,...,...,...,...,...,...
82,S116,575950,31589.0,34301.189061,4344.0,4564.657436,0.922066,0.956432,103.75400,1.28100,-7.345081,-7.629330
83,S207,260944,14996.0,17181.326498,2099.0,2354.268458,0.876405,0.910566,103.95836,1.32485,15.397960,-2.780836
84,S104,544305,32442.0,32424.554578,4140.0,4297.362220,1.000530,0.967198,103.78538,1.44387,-3.852829,10.379206
85,S109,535198,33563.0,32338.730006,4590.0,4308.453008,1.037281,1.058553,103.84920,1.37640,3.249642,2.919048


Then, we construct time related features; we wrap it as a function so that it can be reused in both training and prediction sections. We decided to use sin/cos to encode time as it preserves the temporal dependency of hours within a day (23:00 and 00:00 has a gap numerically). 

Here we explain how the time features are generated in details. We first represent the date of a specific observation into a fraction of the entire year, so April 4 is converted to aorund 0.284. 

Then, we convert the timestamp in a day and in a year to be encoded by sin/cos functions. So h=16 under k=1, 2, 3 has sin=-0.866, 0.866, 0, denoted as t0_hod_s1, t0_hod_s2, t0_hod_s3. The combination of sin/cos can locate a specific point on the unit circle.

Finally, we classify each observation into three categories of seasonality, denoted by 0, 1, and 2.

In [6]:
import math
def add_time_features(df, ts_col="ts"):
    ts = F.col(ts_col)
    h = F.hour(ts) + F.minute(ts) / 60.0
    year = F.year(ts)
    days_in_year = F.datediff(F.make_date(year + 1, F.lit(1), F.lit(1)),
                              F.make_date(year, F.lit(1), F.lit(1)))
    d = (F.dayofyear(ts) - 1 + h / 24.0) / days_in_year

    for k in (1, 2, 3):
        angle = 2 * math.pi * k * h / 24                     
        df = (df.withColumn(f"t0_hod_s{k}", F.sin(angle))
                .withColumn(f"t0_hod_c{k}", F.cos(angle)))
    for k in (1, 2):
        angle = 2 * math.pi * k * d                      
        df = (df.withColumn(f"t0_doy_s{k}", F.sin(angle))
                .withColumn(f"t0_doy_c{k}", F.cos(angle)))

    month = F.month(ts)
    df = df.withColumn("t0_season",
                       F.when(month.isin(12, 1, 2, 3), 0.0)
                        .when(month.isin(6, 7, 8, 9), 2.0)
                        .otherwise(1.0))
    return df

In [7]:
# Use a small sample to check first
test = labels.filter((F.col("station_id") == "S07") & F.col("slot").isin(656247, 660864))
add_time_features(test).select("slot", "ts", *[c for c in add_time_features(test).columns if c.startswith("t0_")]).show(truncate=False)

+------+-------------------+-------------------+-------------------+------------------+-------------------+----------------------+---------+------------------+--------------------+--------------------+-------------------+---------+
|slot  |ts                 |t0_hod_s1          |t0_hod_c1          |t0_hod_s2         |t0_hod_c2          |t0_hod_s3             |t0_hod_c3|t0_doy_s1         |t0_doy_c1           |t0_doy_s2           |t0_doy_c2          |t0_season|
+------+-------------------+-------------------+-------------------+------------------+-------------------+----------------------+---------+------------------+--------------------+--------------------+-------------------+---------+
|660864|2023-04-14 16:00:00|-0.8660254037844384|-0.5000000000000004|0.8660254037844392|-0.4999999999999992|-4.898587196589413E-16|1.0      |0.9772437869053068|-0.21211926115979055|-0.41458446010269895|-0.9100108380900491|1.0      |
+------+-------------------+-------------------+-------------------+----

In [8]:
tf = add_time_features(labels)
tf.agg(F.min("t0_hod_s1"), F.max("t0_hod_s1"),
       F.max(F.col("t0_hod_s1")**2 + F.col("t0_hod_c1")**2 - 1)).show()     
tf.groupBy("t0_season").count().orderBy("t0_season").show()                

+--------------+--------------+------------------------------------------------------+
|min(t0_hod_s1)|max(t0_hod_s1)|max(((POWER(t0_hod_s1, 2) + POWER(t0_hod_c1, 2)) - 1))|
+--------------+--------------+------------------------------------------------------+
|          -1.0|           1.0|                                  2.220446049250313...|
+--------------+--------------+------------------------------------------------------+

+---------+--------+
|t0_season|   count|
+---------+--------+
|      0.0|15382324|
|      1.0|16067629|
|      2.0|16061813|
+---------+--------+



In [9]:
spark.stop()